# Notebook 04 — Cell Type Annotation and Tumor Subcluster Identification

**Project:** RetinoblastomaAtlas  
**Input:** `data/processed/04_scvi_integrated.h5ad`  
**Output:** `data/processed/05_annotated.h5ad`, UMAP figures, cell type proportions

---

## Scientific Background

Cell type annotation is the **interpretive keystone** of the atlas. Every downstream analysis — trajectory inference, cell-cell communication, differential abundance — depends on correct and reproducible labels.

Retinoblastoma has a complex cellular composition:

**Malignant compartment:**
- **Cone precursor-like (CP)**: ARR3, RXRG, THRB, PRDM1, CRX — the primary cells of origin and the cells undergoing invasion
- **Retinoma-like (RL)**: quiescent pre-malignant cells, CDKN2A+, RBL2+, low Ki67
- **MKI67-high proliferating**: MKI67, TOP2A, BIRC5, UBE2C — elevated in extraocular samples (Liu et al. 2024)

**Tumour microenvironment:**
- **Müller glia / Astrocyte-like**: GFAP, VIM, SOX9, RLBP1
- **Microglia / TAM**: AIF1, CD68, CX3CR1 — key immunosuppressive actors
- **Vascular endothelium**: PECAM1, CDH5, VWF
- **Pericyte**: PDGFRB, ACTA2, RGS5
- **T cell / NK cell / B cell / Plasma cell**: standard lymphoid markers

## Annotation Strategy

1. **Gene set scoring** (`sc.tl.score_genes`): per-cell enrichment scores for each cell type's marker set
2. **Label assignment**: highest-scoring cell type with a minimum confidence threshold
3. **Fine subclustering**: higher-resolution clustering within the malignant and TAM compartments
4. **Differential marker genes**: Wilcoxon test to generate cluster-level evidence

**References:**  
- Liu Y et al. Single-cell transcriptomics for RB local extension. *Commun Biol* 2024. https://doi.org/10.1038/s42003-023-05732-y  
- Wan W et al. *Sci Rep* 2025. https://doi.org/10.1038/s41598-025-23779-1

## Parameters

In [ ]:
CP_SUBCLUSTER_RES  = 1.5   # Leiden resolution for cone precursor fine subclustering
TAM_SUBCLUSTER_RES = 1.0   # Leiden resolution for TAM fine subclustering
MIN_SCORE_THRESHOLD = 0.1  # Minimum score to assign a broad cell type label

## Setup

In [ ]:
import warnings
from pathlib import Path

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

warnings.filterwarnings("ignore")
sc.settings.verbosity = 1

ROOT     = Path("..").resolve()
IN_H5AD  = ROOT / "data" / "processed" / "04_scvi_integrated.h5ad"
OUT_H5AD = ROOT / "data" / "processed" / "05_annotated.h5ad"
FIG_DIR  = ROOT / "results" / "figures"
TAB_DIR  = ROOT / "results" / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

## Marker Gene Dictionaries

Markers compiled from Liu et al. 2024 (Commun Biol), Yang et al. 2021 (Cell Death Dis), and the Human Retina Cell Atlas.

In [ ]:
BROAD_MARKERS = {
    "Cone_precursor": ["ARR3", "RXRG", "THRB", "PRDM1", "CRX", "RCVRN", "OPN1MW", "OPN1LW", "GNB3"],
    "Retinoma_like":  ["CDKN2A", "RBL2", "GADD45A"],
    "MKI67_high_RB":  ["MKI67", "TOP2A", "BIRC5", "UBE2C", "CENPF", "NUSAP1"],
    "Muller_glia":    ["GFAP", "VIM", "SOX9", "RLBP1", "CLU", "GLUL"],
    "Microglia_TAM":  ["AIF1", "CX3CR1", "PTPRC", "CD68", "CSF1R"],
    "Endothelium":    ["PECAM1", "CDH5", "VWF", "PLVAP", "KDR"],
    "Pericyte":       ["PDGFRB", "ACTA2", "RGS5", "CSPG4", "NOTCH3"],
    "Fibroblast":     ["DCN", "COL1A1", "LUM", "POSTN", "THY1"],
    "T_cell":         ["CD3D", "CD3E", "CD8A", "CD4", "TRAC"],
    "NK_ILC":         ["GNLY", "NKG7", "KLRD1", "NCR1", "FCGR3A"],
    "B_Plasma":       ["CD79A", "MS4A1", "MZB1", "JCHAIN", "IGKC"],
}

CP_FINE_MARKERS = {
    "CP_mature":        ["ARR3", "OPN1LW", "OPN1MW", "GUCA1C"],
    "CP_intermediate":  ["RXRG", "THRB", "CRX", "PRDM1"],
    "CP_immature":      ["CCND1", "E2F1", "MYCN", "LIN28B"],
    "CP_SOX4_invasive": ["SOX4", "TWIST1", "SNAI2"],
    "CP_proliferating": ["MKI67", "TOP2A", "UBE2C", "BIRC5"],
}

TAM_FINE_MARKERS = {
    "M1_like_TAM":    ["CXCL10", "CXCL9", "IDO1", "GBP1", "STAT1"],
    "M2_like_TAM":    ["CD163", "MRC1", "MIF", "CCL18", "FOLR2", "TREM2"],
    "Transiting_TAM": ["SPP1", "FN1", "MMP9", "HMOX1"],
}

print(f"Defined {len(BROAD_MARKERS)} broad cell types")
print(f"Defined {len(CP_FINE_MARKERS)} cone precursor fine subcategories")
print(f"Defined {len(TAM_FINE_MARKERS)} TAM subcategories")

## Step 1 — Load scVI-Integrated Atlas

In [ ]:
print("Step 1 — Loading scVI-integrated atlas")
adata = sc.read_h5ad(IN_H5AD)
print(f"  {adata.n_obs:,} cells × {adata.n_vars:,} genes")

## Step 2 — Score Broad Cell Type Markers

`sc.tl.score_genes()` computes a per-cell enrichment score for each gene list relative to a control set of randomly sampled genes with similar expression levels (Tirosh et al. 2016, *Science*). This is analogous to AUCell but computationally cheaper and sufficient for broad annotation.

Resulting scores are stored in `adata.obs` and used to assign broad cell type labels.

In [ ]:
def score_cell_types(adata, marker_dict):
    for ct, genes in marker_dict.items():
        present = [g for g in genes if g in adata.var_names]
        if len(present) < 2:
            print(f"  {ct}: only {len(present)} markers found — skipping")
            continue
        sc.tl.score_genes(adata, present, score_name=f"score_{ct}")
        print(f"  Scored: {ct} ({len(present)}/{len(genes)} markers found)")
    return adata

print("Step 2 — Scoring broad cell type markers")
adata = score_cell_types(adata, BROAD_MARKERS)

## Step 3 — Assign Broad Cell Type Labels

In [ ]:
print("Step 3 — Assigning broad cell type labels")
score_cols    = [f"score_{ct}" for ct in BROAD_MARKERS if f"score_{ct}" in adata.obs.columns]
score_matrix  = adata.obs[score_cols].copy()
score_matrix.columns = [c.replace("score_", "") for c in score_cols]
max_scores    = score_matrix.max(axis=1)
max_types     = score_matrix.idxmax(axis=1)
max_types[max_scores < MIN_SCORE_THRESHOLD] = "Unknown"
adata.obs["cell_type_broad"] = pd.Categorical(max_types)

print("  Broad cell type distribution:")
print(adata.obs["cell_type_broad"].value_counts().to_string())

In [ ]:
# UMAP coloured by broad cell type
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sc.pl.umap(adata, color="cell_type_broad", ax=axes[0], show=False, frameon=False,
           size=2, legend_loc="right margin", title="Broad cell type")
sc.pl.umap(adata, color="disease_stage" if "disease_stage" in adata.obs.columns else "dataset",
           ax=axes[1], show=False, frameon=False, size=2, title="Disease stage")
plt.tight_layout()
fig.savefig(FIG_DIR / "umap_cell_types_broad.pdf", dpi=200, bbox_inches="tight")
plt.show()

## Step 4 — Fine Subclustering: Cone Precursor Cells

Broad Leiden clustering uses a resolution tuned for distinguishing major cell types. Within-compartment biology requires higher-resolution clustering on the subset. Liu et al. (2024) identified 10 cone precursor subpopulations; Wan et al. (2025) found that CP4 specifically shows elevated TGF-β in extraocular samples.

Fine subclustering on the scVI latent space ensures that velocity neighbours (notebook 07) are biologically meaningful.

In [ ]:
def fine_subcluster(adata, cell_type, broad_key="cell_type_broad", resolution=1.5, key_added=None):
    if key_added is None:
        key_added = f"leiden_fine_{cell_type.lower()}"
    mask   = adata.obs[broad_key] == cell_type
    n_cells = mask.sum()
    print(f"  Fine subclustering {cell_type}: {n_cells:,} cells, resolution={resolution}")
    if n_cells < 50:
        print(f"  Too few cells ({n_cells}) — skipped")
        return adata
    adata_sub = adata[mask].copy()
    sc.pp.neighbors(adata_sub, use_rep="X_scVI", n_neighbors=20)
    sc.tl.leiden(adata_sub, resolution=resolution, key_added="leiden_fine")
    adata.obs[key_added] = "Other"
    adata.obs.loc[mask, key_added] = cell_type + "_" + adata_sub.obs["leiden_fine"].astype(str)
    adata.obs[key_added] = pd.Categorical(adata.obs[key_added])
    n_sub = adata_sub.obs["leiden_fine"].nunique()
    print(f"  → {n_sub} {cell_type} subclusters identified")
    return adata


print("Step 4 — Fine subclustering: Cone precursor-like cells")
adata = fine_subcluster(adata, "Cone_precursor", resolution=CP_SUBCLUSTER_RES)
adata = score_cell_types(adata, CP_FINE_MARKERS)

print("\nStep 5 — Fine subclustering: Microglia/TAM cells")
adata = fine_subcluster(adata, "Microglia_TAM", resolution=TAM_SUBCLUSTER_RES)
adata = score_cell_types(adata, TAM_FINE_MARKERS)

## Step 5 — Disease Stage Metadata and Cell Type Proportions

Changes in cell type proportions between intraocular and extraocular tumours reflect the compositional remodelling of the tumour and microenvironment during invasion. An increase in the proportion of MKI67-high proliferating cells or SOX4-high invasive cone precursors in extraocular samples would support the dedifferentiation model.

In [ ]:
# Add disease stage if not present
if "disease_stage" not in adata.obs.columns:
    stage_map = {"S1_in1": "intraocular", "S2_in2": "intraocular",
                 "S3_ex1": "extraocular",  "S4_ex2": "extraocular"}
    adata.obs["disease_stage"] = adata.obs["sample_id"].map(
        lambda x: stage_map.get(x, "intraocular")
    )

# Cell type proportions per sample
prop = (
    adata.obs.groupby(["sample_id", "disease_stage", "cell_type_broad"])
    .size().reset_index(name="n_cells")
)
total = prop.groupby("sample_id")["n_cells"].transform("sum")
prop["proportion"] = prop["n_cells"] / total
prop.to_csv(TAB_DIR / "cell_type_proportions.csv", index=False)

# Stacked bar chart
stage_prop = prop.groupby(["disease_stage", "cell_type_broad"])["proportion"].mean().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(12, 5))
stage_prop.T.plot(kind="bar", ax=ax, colormap="tab20", edgecolor="none", width=0.8)
ax.set_xlabel("Cell type", fontsize=11)
ax.set_ylabel("Mean proportion", fontsize=11)
ax.set_title("Cell type composition: intraocular vs. extraocular", fontsize=11)
ax.legend(title="Stage", bbox_to_anchor=(1, 1), fontsize=9)
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
fig.savefig(FIG_DIR / "cell_type_proportions_bar.pdf", dpi=200, bbox_inches="tight")
plt.show()

## Step 6 — Marker Gene Dot Plot

This is the primary evidence figure for cell type annotation. It shows that each annotated cluster expresses its expected markers and not the markers of other cell types (specificity check). Dot size = fraction of cells expressing the gene; colour = z-scored expression.

In [ ]:
selected_markers = {ct: genes[:4] for ct, genes in BROAD_MARKERS.items()}
all_genes = [g for gs in selected_markers.values() for g in gs if g in adata.var_names]

fig, ax = plt.subplots(figsize=(18, 8))
sc.pl.dotplot(
    adata, var_names=all_genes, groupby="cell_type_broad",
    standard_scale="var", cmap="RdBu_r", dot_min=0.05, dot_max=1.0,
    ax=ax, show=False,
)
plt.suptitle("Canonical marker genes per cell type\n(dot size = % expressing, colour = scaled expression)",
             fontsize=10, y=1.01)
plt.tight_layout()
fig.savefig(FIG_DIR / "dotplot_marker_genes.pdf", dpi=200, bbox_inches="tight")
plt.show()

## Step 7 — Differential Marker Genes (Wilcoxon)

The Wilcoxon rank-sum test is the recommended DE test for scRNA-seq data in Scanpy (Zimmermann et al. 2021) because it is non-parametric and robust to the zero-inflation of count data. We run it on the scVI batch-corrected expression layer.

In [ ]:
print("Step 7 — Computing Leiden cluster marker genes (Wilcoxon)")
layer = "scvi_normalized" if "scvi_normalized" in adata.layers else None
sc.tl.rank_genes_groups(
    adata, groupby="cell_type_broad", method="wilcoxon",
    n_genes=50, use_raw=False, layer=layer,
)
marker_genes_df = sc.get.rank_genes_groups_df(adata, group=None)
marker_genes_df.to_csv(TAB_DIR / "marker_genes_per_cell_type.csv", index=False)
print(f"  Saved {len(marker_genes_df):,} marker gene entries")

## Step 8 — Save Annotated Atlas

In [ ]:
print(f"Saving annotated atlas → {OUT_H5AD.name}")
adata.write_h5ad(OUT_H5AD, compression="gzip")
size_mb = OUT_H5AD.stat().st_size / 1e6
print(f"  {adata.n_obs:,} cells × {adata.n_vars:,} genes | {size_mb:.0f} MB")
print("\n  New columns added to .obs:")
print("    cell_type_broad, disease_stage, score_* (per cell type)")
print("    leiden_fine_cone_precursor, leiden_fine_microglia_tam")
print(f"\n  Next: Run notebook 05_copy_number_variation.ipynb")